In [4]:
import requests
import pandas as pd
import json
from datetime import datetime, timedelta
import random

In [5]:
# 1. FETCH DATA
def fetch_products():
    """Get all products from DummyJSON"""
    url = "https://dummyjson.com/products?limit=100"
    response = requests.get(url)
    data = response.json()
    return data['products']

def fetch_carts():
    """Get all shopping carts"""
    url = "https://dummyjson.com/carts?limit=100"
    response = requests.get(url)
    data = response.json()
    return data['carts']

def fetch_users():
    """Get user information"""
    url = "https://dummyjson.com/users?limit=100"
    response = requests.get(url)
    data = response.json()
    return data['users']

In [ ]:
random.seed(42) 

products = fetch_products()
carts = fetch_carts()
users = fetch_users()

print(f"✅ Products: {len(products)}")
print(f"✅ Carts: {len(carts)}")
print(f"✅ Users: {len(users)}")

✅ Products: 100
✅ Carts: 50
✅ Users: 100


In [7]:
# Preview data
print("\n🔍 Sample Product:")
print(json.dumps(products[0], indent=2))


🔍 Sample Product:
{
  "id": 1,
  "title": "Essence Mascara Lash Princess",
  "description": "The Essence Mascara Lash Princess is a popular mascara known for its volumizing and lengthening effects. Achieve dramatic lashes with this long-lasting and cruelty-free formula.",
  "category": "beauty",
  "price": 9.99,
  "discountPercentage": 10.48,
  "rating": 2.56,
  "stock": 99,
  "tags": [
    "beauty",
    "mascara"
  ],
  "brand": "Essence",
  "sku": "BEA-ESS-ESS-001",
  "weight": 4,
  "dimensions": {
    "width": 15.14,
    "height": 13.08,
    "depth": 22.99
  },
  "warrantyInformation": "1 week warranty",
  "shippingInformation": "Ships in 3-5 business days",
  "availabilityStatus": "In Stock",
  "reviews": [
    {
      "rating": 3,
      "comment": "Would not recommend!",
      "date": "2025-04-30T09:41:02.053Z",
      "reviewerName": "Eleanor Collins",
      "reviewerEmail": "eleanor.collins@x.dummyjson.com"
    },
    {
      "rating": 4,
      "comment": "Very satisfied!",
    

In [8]:
# 2. ADD BUSINESS METRICS (Still on your computer)

import random
from datetime import datetime, timedelta

def enrich_cart_data(carts, products, users):
    """Add business intelligence to raw cart data"""
    
    enriched_carts = []
    
    for cart in carts:
        # Calculate cart value
        cart_total = 0
        product_count = 0
        product_categories = []
        
        for item in cart['products']:
            # Find product details
            product = next((p for p in products if p['id'] == item['id']), None)
            if product:
                # Calculate discounted price
                discount_amount = product['price'] * (product['discountPercentage'] / 100)
                final_price = product['price'] - discount_amount
                cart_total += final_price * item['quantity']
                product_count += item['quantity']
                product_categories.append(product['category'])
        
        # Find user details
        user = next((u for u in users if u['id'] == cart['userId']), None)
        
        # BUSINESS LOGIC: Simulate cart abandonment (realistic scenario)
        # In real world, this would come from actual user behavior data
        
        # Factors that increase abandonment:
        # 1. High cart value (> $500)
        # 2. Many items (decision fatigue)
        # 3. Mobile users (harder to checkout)
        # 4. Short session time
        
        abandonment_probability = 0.3  # Base 30% abandonment
        
        if cart_total > 500:
            abandonment_probability += 0.2
        if product_count > 5:
            abandonment_probability += 0.15
        
        is_abandoned = random.random() < abandonment_probability
        
        # Session data (simulated but realistic)
        session_duration = random.randint(2, 45) if not is_abandoned else random.randint(1, 10)
        device_type = random.choice(['Desktop', 'Mobile', 'Tablet'])
        
        if device_type == 'Mobile':
            abandonment_probability += 0.15
            is_abandoned = random.random() < abandonment_probability
        
        # Abandonment reasons (from real e-commerce research)
        abandonment_reasons = [
            "High shipping cost",
            "Just browsing",
            "Found better price elsewhere",
            "Checkout too complicated",
            "Wanted to compare options",
            "Unexpected costs at checkout",
            "Required to create account",
            "Payment security concerns"
        ]
        
        enriched_cart = {
            # Original data
            "cart_id": cart['id'],
            "user_id": cart['userId'],
            "total_products": cart['totalProducts'],
            "total_quantity": cart['totalQuantity'],
            
            # Calculated metrics
            "cart_value": round(cart_total, 2),
            "product_count": product_count,
            "avg_product_price": round(cart_total / product_count, 2) if product_count > 0 else 0,
            
            # User information
            "user_name": f"{user['firstName']} {user['lastName']}" if user else "Unknown",
            "user_age": user['age'] if user else None,
            "user_gender": user['gender'] if user else None,
            
            # Behavioral data
            "device_type": device_type,
            "session_duration_minutes": session_duration,
            "timestamp": (datetime.now() - timedelta(hours=random.randint(1, 72))).isoformat(),
            
            # Business outcomes
            "abandoned": is_abandoned,
            "abandonment_reason": random.choice(abandonment_reasons) if is_abandoned else None,
            
            # Analytics fields
            "cart_size_category": (
                "Small" if cart_total < 100 else
                "Medium" if cart_total < 300 else
                "Large" if cart_total < 600 else
                "Extra Large"
            ),
            "primary_category": max(set(product_categories), key=product_categories.count) if product_categories else "Unknown",
            
            # Revenue metrics
            "revenue": 0 if is_abandoned else round(cart_total, 2),
            "potential_revenue": round(cart_total, 2) if is_abandoned else 0
        }
        
        enriched_carts.append(enriched_cart)
    
    return enriched_carts

# Enrich the data
print("🔧 Enriching cart data with business metrics...")
enriched_carts = enrich_cart_data(carts, products, users)

# Convert to DataFrame for analysis
df_carts = pd.DataFrame(enriched_carts)

print(f"\n✅ Enriched {len(df_carts)} carts")
print(f"📊 Abandonment Rate: {(df_carts['abandoned'].sum() / len(df_carts) * 100):.1f}%")
print(f"💰 Total Revenue: ${df_carts['revenue'].sum():,.2f}")
print(f"😢 Lost Revenue (Abandoned): ${df_carts['potential_revenue'].sum():,.2f}")

# Preview enriched data
print("\n🔍 Sample Enriched Cart:")
print(df_carts.head(3).to_string())

# Quick business insights (local analysis)
print("\n📈 QUICK INSIGHTS:")
print("\n1. Abandonment by Device:")
print(df_carts.groupby('device_type')['abandoned'].agg(['count', 'sum', lambda x: f"{(x.sum()/len(x)*100):.1f}%"]))

print("\n2. Revenue by Cart Size:")
print(df_carts.groupby('cart_size_category')['revenue'].sum().sort_values(ascending=False))

print("\n3. Top Abandonment Reasons:")
print(df_carts[df_carts['abandoned'] == True]['abandonment_reason'].value_counts().head())

🔧 Enriching cart data with business metrics...

✅ Enriched 50 carts
📊 Abandonment Rate: 38.0%
💰 Total Revenue: $108,755.63
😢 Lost Revenue (Abandoned): $76,243.23

🔍 Sample Enriched Cart:
   cart_id  user_id  total_products  total_quantity  cart_value  product_count  avg_product_price     user_name  user_age user_gender device_type  session_duration_minutes                   timestamp  abandoned        abandonment_reason cart_size_category     primary_category   revenue  potential_revenue
0        1       33               4              15     4361.33              7             623.05  Carter Baker      32.0        male      Tablet                         1  2026-04-04T22:14:56.201569       True  Checkout too complicated        Extra Large              laptops      0.00            4361.33
1        2      142               5              20        0.00              0               0.00       Unknown       NaN        None      Tablet                         2  2026-04-03T12:14:56.201710  

In [9]:
# 3. SAVE DATA TO JSON FILE

print("💾 Saving data to data.json...")

df_carts.to_json("data.json", orient="records", indent=2)

print("✅ Data saved successfully!")

💾 Saving data to data.json...
✅ Data saved successfully!


In [10]:
df_carts

,cart_id,user_id,total_products,total_quantity,cart_value,product_count,avg_product_price,user_name,user_age,user_gender,device_type,session_duration_minutes,timestamp,abandoned,abandonment_reason,cart_size_category,primary_category,revenue,potential_revenue
0,1,33,4,15,4361.33,7,623.05,Carter Baker,32.0,male,Tablet,1,2026-04-04T22:14:56.201569,True,Checkout too complicated,Extra Large,laptops,0.00,4361.33
1,2,142,5,20,0.00,0,0.00,Unknown,NaN,None,Tablet,2,2026-04-03T12:14:56.201710,True,Just browsing,Small,Unknown,0.00,0.00
2,3,108,5,13,13316.81,5,2663.36,Unknown,NaN,None,Desktop,4,2026-04-05T22:14:56.201821,False,None,Extra Large,kitchen-accessories,13316.81,0.00
3,4,87,4,17,31.63,7,4.52,Hunter Gordon,34.0,male,Tablet,9,2026-04-06T06:14:56.201895,True,Checkout too complicated,Small,groceries,0.00,31.63
4,5,134,6,20,6588.62,11,598.97,Unknown,NaN,None,Mobile,36,2026-04-04T22:14:56.201982,True,High shipping cost,Extra Large,laptops,0.00,6588.62
5,6,150,4,14,31769.07,3,10589.69,Unknown,NaN,None,Tablet,12,2026-04-04T03:14:56.202061,False,None,Extra Large,mens-watches,31769.07,0.00
6,7,86,3,11,0.00,0,0.00,Nora Mills,27.0,female,Desktop,11,2026-04-04T14:14:56.202122,False,None,Small,Unknown,0.00,0.00
7,8,23,4,14,179.65,7,25.66,Chloe Morales,40.0,female,Desktop,7,2026-04-04T12:14:56.202168,True,Unexpected costs at checkout,Medium,groceries,0.00,179.65
8,9,194,4,13,92.89,2,46.45,Unknown,NaN,None,Tablet,4,2026-04-03T23:14:56.202238,False,None,Small,home-decoration,92.89,0.00
9,10,160,2,8,3724.18,3,1241.39,Unknown,NaN,None,Desktop,26,2026-04-03T11:14:56.202289,False,None,Extra Large,mens-watches,3724.18,0.00


In [11]:
import json
import pandas as pd

# Load JSON again
with open("data.json") as f:
    data_from_file = json.load(f)

df_from_json = pd.DataFrame(data_from_file)

# Compare
print(df_from_json.equals(df_carts))

True
